In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import re
import torch

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# test_str = "### Instruction:\nWrite a Python function that computes the factorial of a number.\n### Response:\n Um 1.23! or maybe 30 million?\n"
test_str = '<<100*12=1200>>\n### 1200\n\n\n### 1200\n\n### 1200\n### 1200\n### 1200\n### 1200\n### 1'

regxp = r'#+\s*\d+\.?\d*'  # matches integers and decimals

tokens = tokenizer.encode(test_str)


def match_regexp_token_indices(tokens: torch.Tensor, regex_pattern: str, tokenizer):
    """
    Find start and end indices (0-based, end exclusive) of the minimal token span
    where the decoded text contains a regex match of maximum length.
    Returns (start, end) or (None, None) if no match.
    Assumes tokens is 1D tensor/list of token IDs.
    """
    tokens_list = tokens.tolist() if isinstance(tokens, torch.Tensor) else tokens
    max_match_len = 0
    candidate_end = len(tokens_list)
    
    # Forward pass: find end of longest match (preferring rightmost if ties)
    for i in range(len(tokens_list)):
        curr_str = tokenizer.decode(tokens_list[:i+1])
        match = re.search(regex_pattern, curr_str)
        if match and len(match.group(0)) > max_match_len:
            max_match_len = len(match.group(0))
            candidate_end = i + 1
    
    if max_match_len == 0:
        return None, None
    
    # Backward pass: find leftmost start for that max length match
    max_match_len = 0
    for j in range(candidate_end):
        curr_str = tokenizer.decode(tokens_list[j:candidate_end])
        match = re.search(regex_pattern, curr_str)
        if (not match) or len(match.group(0)) < max_match_len:
            candidate_start = j - 1
            break
        elif match:
            candidate_start = j
            max_match_len = len(match.group(0))

    return tokens_list[candidate_start:candidate_end]


matched_tokens = match_regexp_token_indices(tokens, regxp, tokenizer)

# show the token ids
current_str = tokenizer.decode(matched_tokens)
print(f"Token IDs: `{current_str}`")

Token IDs: `### 1200`
